# Public OOF stacking route

This notebook does not retrain base models. It stacks public OOF and test predictions from the Kaggle datasets:

- `cdeotte/s6e10-124-public-oof-and-test-predictions`
- `cdeotte/s6e10-tfm-oof-and-test-predictions`

On Kaggle, attach the competition data and both public prediction datasets, then run all cells.

The local fixed LightGBM notebook remains `main.ipynb`. This notebook targets the higher-scoring public model stack.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

try:
    from cuml.linear_model import LogisticRegression
    USE_CUML = True
except ImportError:
    from sklearn.linear_model import LogisticRegression
    USE_CUML = False

INPUT_ROOT = Path(os.getenv("PUBLIC_OOF_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.getenv("PUBLIC_STACK_OUTPUT", "/kaggle/working"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

competition_candidates = [
    path.parent
    for path in INPUT_ROOT.rglob("train.csv")
    if (path.parent / "test.csv").exists()
]
if not competition_candidates:
    raise FileNotFoundError(
        "Could not find train.csv and test.csv under /kaggle/input. "
        "Attach the competition dataset first."
    )
COMP_DIR = competition_candidates[0]

train = pd.read_csv(COMP_DIR / "train.csv", usecols=["id", "satisfaction"])
test = pd.read_csv(COMP_DIR / "test.csv", usecols=["id"])
sample = pd.read_csv(COMP_DIR / "sample_submission.csv")
y = train["satisfaction"].astype(int).to_numpy()

folds = np.zeros(len(y), dtype=int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (_, valid_idx) in enumerate(cv.split(np.zeros(len(y)), y)):
    folds[valid_idx] = fold

print("Competition:", COMP_DIR)
print("Train:", train.shape, "Test:", test.shape)
print("cuML stacker:", USE_CUML)

In [ ]:
def to_logit(values):
    values = np.clip(values.astype(np.float64), 1e-15, 1 - 1e-15)
    return np.log(values / (1 - values))

oof_files = sorted(INPUT_ROOT.rglob("oof_*.npy"))
if not oof_files:
    raise FileNotFoundError(
        "No public oof_*.npy files found. Attach both public OOF datasets."
    )

names = []
oof_columns = []
test_columns = []
rows = []

for oof_path in oof_files:
    model_name = oof_path.stem[len("oof_"):]
    test_path = oof_path.with_name(f"test_{model_name}.npy")
    if not test_path.exists():
        continue

    oof = np.load(oof_path)
    test_pred = np.load(test_path)
    if oof.shape != (len(train),) or test_pred.shape != (len(test),):
        print("Skipping shape mismatch:", oof_path, oof.shape, test_pred.shape)
        continue
    if not np.isfinite(oof).all() or not np.isfinite(test_pred).all():
        print("Skipping non-finite:", oof_path)
        continue

    already_logit = bool(
        (oof.min() < 0 or oof.max() > 1) or
        (test_pred.min() < 0 or test_pred.max() > 1)
    )
    if not already_logit:
        oof = to_logit(oof)
        test_pred = to_logit(test_pred)

    unique_name = f"{oof_path.parent.name}:{model_name}"
    names.append(unique_name)
    oof_columns.append(np.clip(oof, -30, 30).astype(np.float32))
    test_columns.append(np.clip(test_pred, -30, 30).astype(np.float32))
    rows.append(
        {
            "model": unique_name,
            "oof_auc": roc_auc_score(y, oof_columns[-1]),
            "already_logit": already_logit,
        }
    )

if not oof_columns:
    raise ValueError("No usable public OOF/test prediction pairs were found.")

X_oof = np.stack(oof_columns, axis=1)
X_test = np.stack(test_columns, axis=1)
model_table = pd.DataFrame(rows).sort_values("oof_auc", ascending=False)

print("Loaded models:", len(names))
print("X_oof:", X_oof.shape, "X_test:", X_test.shape)
print(model_table.head(15).to_string(index=False))
model_table.to_csv(WORK_ROOT / "public_model_table.csv", index=False)

In [ ]:
oof_stack = np.zeros(len(y), dtype=np.float64)
test_stack = np.zeros(len(test), dtype=np.float64)
coefs = []

for fold in range(5):
    train_mask = folds != fold
    valid_mask = folds == fold

    if USE_CUML:
        model = LogisticRegression(
            penalty="l2",
            C=1.0,
            fit_intercept=True,
            solver="qn",
            max_iter=2000,
            tol=1e-7,
            output_type="numpy",
            verbose=False,
        )
    else:
        model = LogisticRegression(
            penalty="l2",
            C=1.0,
            fit_intercept=True,
            solver="lbfgs",
            max_iter=1000,
            tol=1e-6,
            n_jobs=-1,
        )

    model.fit(X_oof[train_mask], y[train_mask])
    oof_stack[valid_mask] = model.predict_proba(X_oof[valid_mask])[:, 1]
    test_stack += model.predict_proba(X_test)[:, 1] / 5
    coefs.append(np.ravel(model.coef_))
    print(
        f"Fold {fold + 1}/5 AUC: "
        f"{roc_auc_score(y[valid_mask], oof_stack[valid_mask]):.6f}",
        flush=True,
    )

stack_auc = roc_auc_score(y, oof_stack)
print("Stack OOF AUC:", stack_auc)

coef = np.mean(coefs, axis=0)
importance = model_table.assign(coefficient=coef, abs_coefficient=np.abs(coef))
importance.sort_values("abs_coefficient", ascending=False).to_csv(
    WORK_ROOT / "stack_coefficients.csv", index=False
)

submission = pd.DataFrame(
    {"id": test["id"].to_numpy(), "satisfaction": np.clip(test_stack, 0, 1)}
)
assert submission.columns.tolist() == sample.columns.tolist()
assert len(submission) == len(sample)
assert np.array_equal(submission["id"].to_numpy(), sample["id"].to_numpy())
assert submission["satisfaction"].between(0, 1).all()

submission_path = WORK_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)
pd.DataFrame({"id": train["id"], "satisfaction": oof_stack}).to_csv(
    WORK_ROOT / "stack_oof.csv", index=False
)

print("Saved:", submission_path)